In [1]:
# %% [markdown]
# # Diagnostic 2: Numerical Gradient Checking Test
# **Objective:** Validate autograd gradients against two-sided finite difference approximations.

# %%
import torch
import torch.nn as nn
import numpy as np


In [2]:

# Use double precision to prevent numerical rounding artifacts
torch.set_default_dtype(torch.float64)
torch.manual_seed(42)

# 1. Minimal 2-layer network
class SimpleMLP(nn.Module):
    def __init__(self, in_features=4, hidden_dim=6, num_classes=2):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden_dim)
        self.act = nn.Tanh()  # Smooth activation preferred for gradient checking
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))

model = SimpleMLP()
criterion = nn.CrossEntropyLoss()


In [3]:

X = torch.randn(2, 4)
y = torch.tensor([0, 1], dtype=torch.long)

# 2. Compute analytical gradients via backward()
output = model(X)
loss = criterion(output, y)
loss.backward()

# Collect analytical gradients
analytical_grads = torch.cat([p.grad.view(-1) for p in model.parameters() if p.grad is not None])

# 3. Compute numerical gradients via symmetric finite differences
eps = 1e-6
numerical_grads = []

for param in model.parameters():
    param_flat = param.data.view(-1)
    for i in range(param_flat.numel()):
        orig_val = param_flat[i].item()

        # f(theta + eps)
        param_flat[i] = orig_val + eps
        loss_pos = criterion(model(X), y).item()

        # f(theta - eps)
        param_flat[i] = orig_val - eps
        loss_neg = criterion(model(X), y).item()

        # Restore original parameter
        param_flat[i] = orig_val

        grad_approx = (loss_pos - loss_neg) / (2.0 * eps)
        numerical_grads.append(grad_approx)

numerical_grads = torch.tensor(numerical_grads)

# 4. Relative difference metric
diff = torch.norm(analytical_grads - numerical_grads) / (
    torch.norm(analytical_grads) + torch.norm(numerical_grads)
)

print(f"Analytical norm: {torch.norm(analytical_grads):.6f}")
print(f"Numerical norm:  {torch.norm(numerical_grads):.6f}")
print(f"Relative error:   {diff.item():.2e}")

# Reset default dtype
torch.set_default_dtype(torch.float32)

print("\n--- Diagnostic Verdict ---")
if diff < 1e-6:
    print("PASSED: Relative difference < 1e-6. Autograd graph matches exact mathematical gradients.")
else:
    print("FAILED: Discrepancy detected. Check custom autograd functions or disconnected computational graph.")

Analytical norm: 0.774725
Numerical norm:  0.774725
Relative error:   1.93e-10

--- Diagnostic Verdict ---
PASSED: Relative difference < 1e-6. Autograd graph matches exact mathematical gradients.


# Diagnostic 2b: Injected Backward-Pass Bug (Custom Tanh)

**The bug we introduce:** replace `nn.Tanh` with a hand-written `torch.autograd.Function`.
The forward pass is correct (`tanh(x)`), but the backward pass uses the wrong derivative:

| | Derivative used in `backward` |
|---|---|
| Correct | `1 - tanh(x)^2` |
| **Buggy** | `1 - tanh(x)` (forgot to square) |

This is a typical custom-op mistake. The forward value is right, so the loss looks normal and nothing crashes.
The finite-difference estimate only depends on the forward pass, so it still sees the true gradient and disagrees with the buggy analytic one.

We run the same check on a **correct** custom tanh as a control, to show the harness only fails when the backward is actually wrong.

In [4]:
# Custom tanh implementations: one correct (control), one with a wrong backward
class CorrectTanhFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        y = torch.tanh(x)
        ctx.save_for_backward(y)
        return y

    @staticmethod
    def backward(ctx, grad_out):
        (y,) = ctx.saved_tensors
        return grad_out * (1 - y ** 2)       # correct: d/dx tanh = 1 - tanh^2


class BuggyTanhFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        y = torch.tanh(x)
        ctx.save_for_backward(y)
        return y

    @staticmethod
    def backward(ctx, grad_out):
        (y,) = ctx.saved_tensors
        return grad_out * (1 - y)            # BUG: should be (1 - y ** 2)


class CorrectTanh(nn.Module):
    def forward(self, x):
        return CorrectTanhFn.apply(x)


class BuggyTanh(nn.Module):
    def forward(self, x):
        return BuggyTanhFn.apply(x)


class SimpleMLPCustomAct(nn.Module):
    def __init__(self, act, in_features=4, hidden_dim=6, num_classes=2):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden_dim)
        self.act = act
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))

In [5]:
# Reusable gradient-check harness (same method as the cell above, plus a per-parameter breakdown)
def gradient_check(model, X, y, eps=1e-6):
    criterion = nn.CrossEntropyLoss()

    # Analytical gradients via backward()
    model.zero_grad()
    criterion(model(X), y).backward()
    analytic = {n: p.grad.detach().clone().view(-1) for n, p in model.named_parameters()}

    # Numerical gradients via symmetric finite differences
    numeric = {}
    for n, p in model.named_parameters():
        flat = p.data.view(-1)
        vals = []
        for i in range(flat.numel()):
            orig = flat[i].item()
            flat[i] = orig + eps
            lp = criterion(model(X), y).item()
            flat[i] = orig - eps
            ln = criterion(model(X), y).item()
            flat[i] = orig
            vals.append((lp - ln) / (2.0 * eps))
        numeric[n] = torch.tensor(vals)

    def rel(a, b):
        return (torch.norm(a - b) / (torch.norm(a) + torch.norm(b))).item()

    per_param = {n: rel(analytic[n], numeric[n]) for n in analytic}
    overall = rel(torch.cat(list(analytic.values())), torch.cat(list(numeric.values())))
    return overall, per_param


def run_case(label, act):
    torch.set_default_dtype(torch.float64)       # double precision, as in the baseline
    try:
        torch.manual_seed(42)
        model = SimpleMLPCustomAct(act)
        X = torch.randn(2, 4)
        y = torch.tensor([0, 1], dtype=torch.long)
        overall, per_param = gradient_check(model, X, y)
    finally:
        torch.set_default_dtype(torch.float32)

    print(f"=== {label} ===")
    for n, e in per_param.items():
        flag = "ok " if e < 1e-6 else "BAD"
        print(f"  [{flag}] {n:<12} relative error: {e:.2e}")
    print(f"  Overall relative error: {overall:.2e}")
    print("  Verdict:", "PASSED" if overall < 1e-6 else "FAILED: Discrepancy detected. Check custom autograd functions or disconnected computational graph.")
    print()
    return overall

err_control = run_case("Control: custom tanh, CORRECT backward", CorrectTanh())
err_buggy = run_case("Buggy: custom tanh, backward uses (1 - y) instead of (1 - y^2)", BuggyTanh())

=== Control: custom tanh, CORRECT backward ===
  [ok ] fc1.weight   relative error: 4.12e-10
  [ok ] fc1.bias     relative error: 8.75e-10
  [ok ] fc2.weight   relative error: 1.16e-10
  [ok ] fc2.bias     relative error: 3.10e-08
  Overall relative error: 1.93e-10
  Verdict: PASSED

=== Buggy: custom tanh, backward uses (1 - y) instead of (1 - y^2) ===
  [BAD] fc1.weight   relative error: 5.02e-01
  [BAD] fc1.bias     relative error: 9.21e-01
  [ok ] fc2.weight   relative error: 1.16e-10
  [ok ] fc2.bias     relative error: 3.10e-08
  Overall relative error: 2.84e-01
  Verdict: FAILED: Discrepancy detected. Check custom autograd functions or disconnected computational graph.



## Takeaways

- **The control passes and the buggy version fails.** The correct custom tanh matches finite differences to about 1e-10, while the buggy backward is off by many orders of magnitude.
- **The check localizes the bug.** `fc2` (after the activation) gets correct gradients, because the loss-to-logits-to-`fc2` path never touches the custom backward. `fc1` (before the activation) is wrong, because its gradient flows through the buggy derivative. The layer where errors begin is the layer just before the broken op.
- **Nothing else would flag this.** The forward pass and loss values are correct, and there is no crash or NaN. Only comparing against an independent estimate of the gradient reveals it.
- **Fix:** return `grad_out * (1 - y ** 2)` from `backward`, or use `nn.Tanh` and let autograd derive it.